# Notebook 10 — Metabolic Networks and Flux Balance Analysis

**Module:** 12 — Systems and Network Biology  
**Tier:** 2 — Working competence  
**Notebook:** 10 of 12  
**Time estimate:** 90 minutes

> FBA is one of the few systems biology methods that makes quantitative, testable
> predictions about living cells without knowing any kinetic parameters. It asks:
> given stoichiometric constraints and a growth objective, what flux distribution
> maximizes growth?

---
## Step 1 — Motivation

Metabolic networks underlie every cellular process — ATP production, biosynthesis,
signal transduction. Flux Balance Analysis (FBA) uses only the stoichiometry of
metabolic reactions (no kinetics required) to predict growth rates, essential genes,
and metabolic engineering strategies. It is the most widely used quantitative
method in systems and synthetic biology.

---
## Step 2 — Intuition

**Metabolic network:** A directed bipartite graph with two node types:
- **Metabolites** (glucose, ATP, NADH, ...)
- **Reactions** (hexokinase, ATP synthase, ...)

Each reaction consumes some metabolites (substrates) and produces others (products).
The stoichiometry is encoded in the **stoichiometric matrix** $S$ ($m \times r$):
$S_{ij}$ = net production of metabolite $i$ by reaction $j$ (negative if consumed).

**Steady-state assumption:** Metabolite concentrations don't change → $S \mathbf{v} = 0$,
where $\mathbf{v}$ is the flux vector (reaction rates).

**Feasible flux space:** A convex polytope in $\mathbb{R}^r$ defined by $S\mathbf{v}=0$
and flux bounds $\mathbf{v}^{lb} \leq \mathbf{v} \leq \mathbf{v}^{ub}$.

**FBA:** Find the vertex of this polytope that maximizes a linear objective (usually
the biomass reaction flux = growth). This is a linear program (LP).

---
## Step 3 — Biological Background

**Genome-scale metabolic models (GEMs):**
- E. coli iJO1366: 1805 genes, 2583 reactions, 1805 metabolites (Orth et al. 2011)
- Human Recon 3D: 3288 genes, 13543 reactions, 4140 metabolites
- Construction: gene-protein-reaction (GPR) associations; gap filling; experimental validation

**FBA predictions that were experimentally validated:**
- Essential genes (flux must be nonzero for growth) — ~80% accuracy vs. deletion screens
- Optimal growth phenotype under many nutrient conditions
- Metabolic engineering: predict gene knockouts that redirect flux to desired product

**Exchange reactions:** Reactions representing import/export across cell membrane.
Setting their bounds defines the nutrient environment (e.g., glucose uptake bound = −10 mmol/gDW/h).

**Shadow prices (dual variables):** The LP dual gives the sensitivity of the objective
to constraint relaxation. Shadow price of a metabolite = marginal growth benefit of
slightly more of that metabolite — identifies "limiting" metabolites.

**Parsimonious FBA (pFBA):** Among all flux distributions achieving maximum growth,
find the one minimizing total flux (enzyme budget). More biologically realistic:
cells tend to use minimal enzyme.

---
## Step 4 — Mathematical Explanation

**FBA as a linear program:**
$$\max_{\mathbf{v}} \; \mathbf{c}^T \mathbf{v}$$
$$\text{subject to:} \quad S \mathbf{v} = \mathbf{0}, \quad \mathbf{v}^{lb} \leq \mathbf{v} \leq \mathbf{v}^{ub}$$

where $\mathbf{c}$ is a vector with 1 in the biomass reaction position and 0 elsewhere.

**Stoichiometric matrix construction:**
- Row $i$: metabolite $i$
- Column $j$: reaction $j$
- $S_{ij} > 0$: metabolite $i$ is produced by reaction $j$
- $S_{ij} < 0$: metabolite $i$ is consumed by reaction $j$
- $S_{ij} = 0$: metabolite $i$ not involved in reaction $j$

**Feasibility:** Not all $S\mathbf{v}=\mathbf{0}, \mathbf{v}^{lb} \leq \mathbf{v} \leq \mathbf{v}^{ub}$
may be feasible — indicates a model gap (missing reactions).

**Null space of S:** The feasible flux space (when $\mathbf{v}^{lb}=\mathbf{0}$, $\mathbf{v}^{ub}=\infty$)
is the right null space of $S$. FBA finds a specific point within this space
by optimizing the linear objective.

In [ ]:
# Step 6 — Python: Toy FBA from scratch using scipy linprog
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog

# ---- Toy metabolic model: simplified glycolysis + biomass ----
# Metabolites: Glc (glucose), G6P, F6P, PEP, PYR, ATP, BIOMASS
# Indices:       0    1    2    3    4    5       6

# Reactions and their stoichiometry (column = reaction):
# R1: Glucose uptake:    Glc_ext → Glc         (exchange)
# R2: Hexokinase:        Glc + ATP → G6P       (consumes 1 ATP, produces 1 G6P)
# R3: PGI:               G6P ↔ F6P             (reversible)
# R4: Lower glycolysis:  F6P → 2 PEP + 2 ATP   (net; simplified)
# R5: Pyruvate kinase:   PEP → PYR + ATP
# R6: ATP synthesis:     ADP → ATP             (simplified)
# R7: Biomass reaction:  G6P + ATP → BIOMASS   (objective)
# R8: ATP maintenance:   ATP → (ATPm, maintenance)

# Metabolites: 0=Glc, 1=G6P, 2=F6P, 3=PEP, 4=PYR, 5=ATP
n_mets = 6  # metabolites
n_rxns = 8  # reactions

# S[i,j] = net stoichiometry of metabolite i in reaction j
S = np.zeros((n_mets, n_rxns))
# R0: Glucose uptake (exchange)
S[0, 0] = 1   # Glc produced
# R1: Hexokinase: Glc + ATP -> G6P
S[0, 1] = -1  # Glc consumed
S[5, 1] = -1  # ATP consumed
S[1, 1] = 1   # G6P produced
# R2: PGI: G6P <-> F6P (reversible)
S[1, 2] = -1  # G6P consumed
S[2, 2] = 1   # F6P produced
# R3: Lower glycolysis: F6P -> 2 PEP + 2 ATP
S[2, 3] = -1  # F6P consumed
S[3, 3] = 2   # PEP produced
S[5, 3] = 2   # ATP produced
# R4: Pyruvate kinase: PEP -> PYR + ATP
S[3, 4] = -1  # PEP consumed
S[4, 4] = 1   # PYR produced
S[5, 4] = 1   # ATP produced
# R5: PYR export/consumption (exchange)
S[4, 5] = -1  # PYR consumed
# R6: Biomass: G6P + ATP -> BIOMASS (sink reaction)
S[1, 6] = -1  # G6P consumed
S[5, 6] = -1  # ATP consumed
# R7: ATP maintenance (minimum ATP consumption)
S[5, 7] = -1  # ATP consumed

print('Stoichiometric matrix S:')
met_names = ['Glc', 'G6P', 'F6P', 'PEP', 'PYR', 'ATP']
rxn_names = ['Glc_uptake', 'Hexokinase', 'PGI', 'LowerGlyc', 'PK', 'PYR_export', 'Biomass', 'ATPmaint']
print(f'     {" ".join([f"{r:<11}" for r in rxn_names])}')
for i, met in enumerate(met_names):
    print(f'{met:<5} {" ".join([f"{S[i,j]:<11.0f}" for j in range(n_rxns)])}')

# ---- FBA via scipy.optimize.linprog ----
# scipy linprog minimizes c@x; we want to maximize biomass (R6), so c[6] = -1
c = np.zeros(n_rxns)
c[6] = -1  # maximize biomass flux

# Equality constraint: S*v = 0
A_eq = S
b_eq = np.zeros(n_mets)

# Flux bounds
vlb = np.zeros(n_rxns)  # all irreversible by default
vub = np.full(n_rxns, 100.0)  # upper bound 100
# PGI is reversible
vlb[2] = -100.0
# Glucose uptake bounded: max 10 mmol/gDW/h
vub[0] = 10.0
# ATP maintenance: at least 1 unit
vlb[7] = 1.0
bounds = list(zip(vlb, vub))

result = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
if result.success:
    v_opt = result.x
    print(f'\nFBA: maximum biomass flux = {v_opt[6]:.4f}')
    print(f'Optimal flux distribution:')
    for i, (name, v) in enumerate(zip(rxn_names, v_opt)):
        if abs(v) > 1e-6:
            print(f'  {name}: {v:.4f}')
else:
    print(f'FBA infeasible: {result.message}')

# ---- Gene essentiality: knock out each reaction ----
print('\nReaction essentiality (flux=0 for each reaction):')
for ko_rxn in range(n_rxns):
    if ko_rxn == 6: continue  # skip biomass itself
    bounds_ko = list(zip(vlb, vub))
    bounds_ko[ko_rxn] = (0, 0)  # force flux to 0
    res_ko = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds_ko, method='highs')
    if not res_ko.success or res_ko.x[6] < 0.001:
        print(f'  ESSENTIAL: {rxn_names[ko_rxn]}')
    else:
        print(f'  dispensable: {rxn_names[ko_rxn]} (growth = {res_ko.x[6]:.3f})')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Stoichiometric matrix heatmap
ax = axes[0]
im = ax.imshow(S, cmap='RdBu_r', aspect='auto', vmin=-3, vmax=3)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(n_rxns))
ax.set_xticklabels(rxn_names, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(n_mets))
ax.set_yticklabels(met_names, fontsize=8)
ax.set_title('A. Stoichiometric matrix S\n(red=produced, blue=consumed)')

# Panel B: Optimal flux distribution
ax = axes[1]
if result.success:
    colors = ['green' if abs(v_opt[i]) > 0.001 else 'lightgrey'
              for i in range(n_rxns)]
    ax.barh(range(n_rxns), v_opt, color=colors, edgecolor='black')
    ax.set_yticks(range(n_rxns))
    ax.set_yticklabels(rxn_names, fontsize=8)
    ax.set_xlabel('Flux (mmol/gDW/h)')
    ax.set_title(f'B. FBA optimal flux distribution\nBiomass={v_opt[6]:.3f}')
    ax.axvline(0, color='black', lw=0.5)

# Panel C: Phenotypic phase plane — biomass vs. glucose uptake
ax = axes[2]
glc_range = np.linspace(0, 10, 30)
biomass_vals = []
for glc_ub in glc_range:
    bounds_scan = list(zip(vlb, vub))
    bounds_scan[0] = (0, glc_ub)  # vary glucose uptake
    r = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds_scan, method='highs')
    biomass_vals.append(r.x[6] if r.success else 0)
ax.plot(glc_range, biomass_vals, 'steelblue', lw=2)
ax.set_xlabel('Maximum glucose uptake (mmol/gDW/h)')
ax.set_ylabel('Biomass flux (au)')
ax.set_title('C. Growth vs. glucose availability\n(phenotypic phase plane scan)')
ax.axvline(10, color='red', ls='--', lw=0.8, label='Current bound')
ax.legend(fontsize=8)

plt.suptitle('Module 12 NB10: Metabolic Networks and Flux Balance Analysis',
               fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('fba_metabolic.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Add a competing pathway: a reaction that consumes G6P to produce a byproduct
   (e.g., G6P → byproduct, irreversible). How does this affect maximum biomass?
2. Compute the null space of S using `np.linalg.svd`. How many degrees of freedom
   does the feasible flux space have (before adding bounds)?
3. Implement parsimonious FBA: after finding maximum biomass $v^*_\text{bio}$, add
   a constraint $v_\text{bio} = v^*_\text{bio}$ and minimize $\sum |v_i|$.
4. (Optional) Install COBRApy and load the E. coli iJO1366 model. Run FBA and
   identify the top 10 reactions by flux in optimal growth conditions.

---
## Step 10 — Quiz

1. What does the stoichiometric matrix $S$ encode?
2. Why is $S\mathbf{v} = 0$ the steady-state constraint?
3. What is the FBA objective, and what does it biologically represent?
4. What are shadow prices and what do they tell you about a metabolite?
5. How would you predict essential genes from FBA?

---
## Step 12 — Reflection

> *[In one paragraph: explain why FBA can predict growth rates without knowing any
> kinetic parameters. What assumption makes this possible, and when would you expect
> it to fail?]*

---
*Next: `11_network_medicine_and_disease_modules.ipynb`*